> ## SOLUTION / ANSWER KEY &mdash; Lab 5.9
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-09-tool-selection.ipynb`](../lab-09-tool-selection.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.9 &mdash; Tool Selection: Pick the Right Tool

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Ask the REAL model to choose a tool from descriptions for each request
- Run it across a suite of tasks (including a digit-but-not-math trap)
- Measure how often the choice is right -- and tune the descriptions

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; Tools turn an LLM into an agent](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-05-09")
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
With more than two tools, the agent must **select** the right one for each sub-goal &mdash; the hardest
part of tool use. A real agent reads each tool's **description** and picks. Here the **real model** does the
selecting; you **measure** how often it's right across a suite. There's **no grader** &mdash; an 8b model
won't always be perfect (watch the *"16th president"* trap: a digit, but not arithmetic), and that's the
point: you inspect the misses and sharpen the descriptions, the goal-driven loop.

In [ ]:
# DEMO -- four tools; the model will choose from their descriptions
print("tools:", ["calculator", "weather", "translate", "lookup"])
print("Trap: 'the 16th president' has a digit but is a LOOKUP, not arithmetic.")

## Build it
Write `select_with_model` (prompt the model with the catalog, parse its pick) and `measure` (accuracy over
the suite). Descriptions are the lever &mdash; the model reads *them*, not your code.

In [ ]:
TOOLSET = ["calculator", "weather", "translate", "lookup"]
CATALOG = ("- calculator: exact arithmetic like 12*8\n"
           "- weather: current weather for a city\n"
           "- translate: translate English text to French\n"
           "- lookup: look up a known fact")

def select_with_model(text):
    prompt = ("Tools:\n" + CATALOG + "\n\nWhich ONE tool answers: '" + text + "'?\nReply with ONLY the tool name.")
    reply = llm_text(prompt).lower()
    return next((t for t in TOOLSET if t in reply), "lookup")

TASKS = [
    {"text": "what is 12 * 8", "tool": "calculator"},
    {"text": "current weather in tokyo", "tool": "weather"},
    {"text": "translate hello to french", "tool": "translate"},
    {"text": "capital of france", "tool": "lookup"},
    {"text": "who was the 16th president", "tool": "lookup"},   # a digit, but NOT arithmetic
]

def measure(picks):
    hits = sum(1 for i, t in enumerate(TASKS) if picks[i] == t["tool"])
    return hits, len(TASKS)

try:
    print("catalog the model will read:\n" + CATALOG)
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run the real model over the whole suite and read where it routes well and where it slips.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        picks = []
        for t in TASKS:
            pick = select_with_model(t["text"])          # REAL model selects from the descriptions
            picks.append(pick)
            print(("ok " if pick == t["tool"] else "XX "), t["text"], "-> model:", pick, "| expected:", t["tool"])
        print("---")
        print("selection accuracy:", measure(picks))
        print("(No grader. If the model misroutes, inspect WHY and sharpen the tool descriptions.)")
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Each pick came from the **real model** reading your **descriptions** &mdash; not a keyword rule you hardcoded.
- If it misroutes (e.g. sends *"16th president"* to calculator, or a terse task to lookup), that's real selection error &mdash; **no grader hides it**.
- The fix is almost always a sharper **description**, not more code &mdash; that's how you'd tune a real agent.

## Your turn (open task &mdash; no grader)
Improve a tool's description to fix a misroute (e.g. clarify calculator is for *arithmetic expressions*,
not any number), and re-run. **What good looks like:** accuracy climbs as your descriptions get clearer.
Add a fifth tool and a task for it and see if the model picks it.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
# Sharpen the calculator description (the misroute lever) and add a 5th tool + task, then re-measure.
CATALOG2 = ("- calculator: arithmetic EXPRESSIONS only, like 12*8 (NOT questions that merely contain a number)\n"
            "- weather: current weather for a city\n"
            "- translate: translate English text to French\n"
            "- lookup: look up a known fact, including history and people\n"
            "- define: give the dictionary definition of a single word")
TOOLSET2 = TOOLSET + ["define"]
def select2(text):
    reply = llm_text("Tools:\n" + CATALOG2 + "\n\nWhich ONE tool answers: '" + text +
                     "'?\nReply with ONLY the tool name.").lower()
    return next((t for t in TOOLSET2 if t in reply), "lookup")
if ollama_up():
    for t in TASKS + [{"text": "define serendipity", "tool": "define"}]:
        pick = select2(t["text"])
        print(("ok " if pick == t["tool"] else "XX "), t["text"], "-> model:", pick, "| expected:", t["tool"])
else:
    print("(start Ollama to re-measure with the sharper descriptions)")

---
### Key takeaway
Selecting the right tool per sub-goal is the core skill of tool use -- and a REAL model selects from descriptions, imperfectly. With no grader, you read the misses and sharpen the words the model reads.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>